In [ ]:
from core.io import load_parquet
from core.validation import validate_play
from core.scoring import score_play
from optimizer.solve import solve_all_gameweeks
from optimizer.pruning import prune_by_score_per_price, prune_by_three_score_per_price
import time
import matplotlib.pyplot as plt

In [ ]:
stats = load_parquet("player_data.parquet")
small_stats = stats[stats["gw"]<=22]

In [ ]:
scores = []
times = []
tresholds = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6]
for treshold in tresholds:
    treshold
    pruned_keys = prune_by_score_per_price(small_stats, threshold=treshold)
    print(len(pruned_keys))
    tic = time.time()
    picks, chips,_ = solve_all_gameweeks(stats=small_stats,pruned_keys=pruned_keys)
    times.append(time.time()-tic)
    errors = validate_play(picks, chips, small_stats)
    if errors:
        print("Invalid play found:")
        for e in errors:
            print(f"  - {e}")
    result = score_play(picks, chips, small_stats)
    scores.append(result['total'])

In [ ]:
plt.plot(tresholds[0:6], scores,'*-')
plt.plot(tresholds[0:6], times[0:6],'*-')

In [ ]:
plt.plot(times[0:6],scores,'*-')
plt.ylabel("score")
plt.xlabel("time (seconds)")

In [ ]:
fig, ax1 = plt.subplots(figsize=(8, 5))

# Left axis
color1 = 'tab:blue'
ax1.set_xlabel('treshold')
ax1.set_ylabel('score', color=color1)
line1, = ax1.plot(tresholds, scores,'*-', color=color1)
ax1.tick_params(axis='y', labelcolor=color1)

# Right axis, sharing the same x-axis
ax2 = ax1.twinx()
color2 = 'tab:red'
ax2.set_ylabel('time (seconds)', color=color2)
line2, = ax2.plot(tresholds, times,'*-', color=color2)
ax2.tick_params(axis='y', labelcolor=color2)

ax1.grid()
# Combined legend (optional, since each axis has its own by default)
#ax1.legend([line1, line2], ['Y1', 'Y2'], loc='upper left')

fig.tight_layout()
plt.show()

In [ ]:
import pandas as pd
results = pd.DataFrame({"threshold": tresholds, "score": scores, "time_s": times})
results["method"] = "score_per_price"   # or "three_score_per_price" for the second experiment
results["through_gw"] = int(small_stats["gw"].max())
results.to_parquet("solutions/threshold_pruning_score_per_price.parquet")

In [ ]:
small_stats = stats[stats["gw"]<=22]
pruned_keys = prune_by_score_per_price(small_stats, threshold=0.6)
print(len(pruned_keys))
picks, chips, _ = solve_all_gameweeks(stats=small_stats,pruned_keys=pruned_keys)


In [ ]:
pruned = prune_by_score_per_price(small_stats, threshold= 0.2)
new_pruned = prune_by_three_score_per_price(small_stats, threshold=0.4)
print(len(pruned))
print(len(new_pruned))
print(sorted(new_pruned)[:10])

In [ ]:
new_pruned = prune_by_three_score_per_price(small_stats, threshold=1.45)
picks, chips, _ = solve_all_gameweeks(stats=small_stats,pruned_keys=new_pruned)

In [ ]:
scores = []
times = []
tresholds = [0.4, 0.6, 0.8, 1, 1.2, 1.45]
for treshold in tresholds:
    treshold
    pruned_keys = prune_by_three_score_per_price(small_stats, threshold=treshold)
    print(len(pruned_keys))
    tic = time.time()
    picks, chips,_ = solve_all_gameweeks(stats=small_stats,pruned_keys=pruned_keys)
    times.append(time.time()-tic)
    errors = validate_play(picks, chips, small_stats)
    if errors:
        print("Invalid play found:")
        for e in errors:
            print(f"  - {e}")
    result = score_play(picks, chips, small_stats)
    scores.append(result['total'])

In [ ]:
fig, ax1 = plt.subplots(figsize=(8, 5))

# Left axis
color1 = 'tab:blue'
ax1.set_xlabel('treshold')
ax1.set_ylabel('score', color=color1)
line1, = ax1.plot(tresholds, scores, '*-', color=color1)
ax1.tick_params(axis='y', labelcolor=color1)

# Right axis, sharing the same x-axis
ax2 = ax1.twinx()
color2 = 'tab:red'
ax2.set_ylabel('time (seconds)', color=color2)
line2, = ax2.plot(tresholds, times,'*-', color=color2)
ax2.tick_params(axis='y', labelcolor=color2)

# Combined legend (optional, since each axis has its own by default)
#ax1.legend([line1, line2], ['Y1', 'Y2'], loc='upper left')

fig.tight_layout()
plt.show()

In [ ]:
results = pd.DataFrame({"threshold": tresholds, "score": scores, "time_s": times})
results["method"] = "three_score_per_price"   # or "three_score_per_price" for the second experiment
results["through_gw"] = int(small_stats["gw"].max())
results.to_parquet("solutions/threshold_pruning_three_score_per_price.parquet")